# AUModel — Phase 1: Train Turkish BPE Tokenizer

This notebook trains a 64k-vocabulary SentencePiece BPE tokenizer on Turkish Wikipedia.

**Estimated total runtime**: 35–55 minutes on Colab CPU (T4 GPU not needed for this phase).

| Step | Action | Est. Time |
|------|--------|-----------|
| 1 | Install deps + mount Drive | < 2 min |
| 2 | Download corpus (`--download`) | 10–15 min |
| 3 | Train tokenizer (`--train`) | 25–35 min |
| 4 | Validate + copy to Drive | < 2 min |

---
## Section 1: Install Dependencies & Mount Google Drive

In [ ]:
# Install required packages
# Expected: all packages install without errors
!pip install sentencepiece>=0.1.99 datasets>=2.18.0 tqdm>=4.66.0 -q
print("[OK] Dependencies installed")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/aumodel_checkpoints/tokenizer'

import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"[OK] Drive mounted. Artifacts will be saved to: {DRIVE_DIR}")

In [ ]:
# Clone the repo on the Colab runtime filesystem (/content/)
# Works the same whether you open this notebook in browser Colab OR via VS Code Colab extension
# (Colab runtime has its own filesystem — separate from your local machine)
import os

REPO_URL = 'https://github.com/aykanugur/AU_MODEL.git'
REPO_BRANCH = '001-turkish-tokenizer'
REPO_DIR = '/content/AU_MODEL'

if not os.path.exists(REPO_DIR):
    !git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}
    print("[OK] Repo cloned")
else:
    print("[OK] Repo already present — pulling latest")
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
!git log --oneline -3
print(f"[OK] Working dir: {os.getcwd()}")

---
## Section 2: Download Turkish Wikipedia Corpus

Downloads `wikimedia/wikipedia 20231101.tr` via HuggingFace, applies NFC normalization,
filters documents < 200 chars, and writes one document per line to
`data/raw/tokenizer_corpus.txt`.

**Expected output**:
```
[INFO] Downloading wikimedia/wikipedia (20231101.tr) ...
[INFO] Wrote 1,180,000 documents (45,000 skipped) -> data/raw/tokenizer_corpus.txt
[INFO] Corpus size: 7400.0 MB [OK]
```
**Expected duration**: ~10–15 minutes (network dependent)

In [ ]:
import os
corpus_path = 'data/raw/tokenizer_corpus.txt'
if os.path.exists(corpus_path):
    size_mb = os.path.getsize(corpus_path) / 1024 / 1024
    print(f"[INFO] Corpus already exists ({size_mb:.0f} MB). Skipping download.")
    SKIP_DOWNLOAD = True
else:
    print("[INFO] Corpus not found — will download.")
    SKIP_DOWNLOAD = False

In [ ]:
if not SKIP_DOWNLOAD:
    !python tokenizer/train_tokenizer.py --download
else:
    print("[SKIP] Download skipped (corpus already present)")

In [ ]:
# Verify corpus
import os
corpus_path = 'data/raw/tokenizer_corpus.txt'
assert os.path.exists(corpus_path), f"Corpus not found: {corpus_path}"
size_mb = os.path.getsize(corpus_path) / 1024 / 1024
print(f"[OK] Corpus: {corpus_path} ({size_mb:.0f} MB)")

# Show first 3 lines
with open(corpus_path, encoding='utf-8') as f:
    for i, line in enumerate(f):
        print(f"  Line {i+1}: {line[:80]}...")
        if i >= 2:
            break

---
## Section 3: Train the Tokenizer

Trains a 64k-vocabulary BPE SentencePiece model with locked parameters:
- `vocab_size=64000`, `model_type=bpe`, `character_coverage=0.9999`
- `normalization_rule_name=identity`, `byte_fallback=True`
- `input_sentence_size=10_000_000`, `random_seed=42`
- Special tokens: `<pad>=0, <unk>=1, <s>=2, </s>=3, [SYSTEM]=4, [USER]=5, [ASSISTANT]=6, [SEP]=7`

**Expected output**:
```
[INFO] Training SentencePiece BPE tokenizer (vocab_size=64,000) ...
[INFO] Model saved: tokenizer/turkish_bpe.model [OK]
[INFO] Vocab saved: tokenizer/turkish_bpe.vocab [OK]
[INFO] Vocab size verified: 64,000 [OK]
```
**Expected duration**: ~25–35 minutes on Colab CPU

In [ ]:
import os
model_path = 'tokenizer/turkish_bpe.model'
if os.path.exists(model_path):
    size_mb = os.path.getsize(model_path) / 1024 / 1024
    print(f"[INFO] Model already exists ({size_mb:.1f} MB). Use --force to retrain.")
    SKIP_TRAIN = True
else:
    print("[INFO] Model not found — will train.")
    SKIP_TRAIN = False

In [ ]:
if not SKIP_TRAIN:
    !python tokenizer/train_tokenizer.py --train
else:
    print("[SKIP] Training skipped (model already present)")
    print("       Pass --force to retrain: !python tokenizer/train_tokenizer.py --train --force")

In [ ]:
# Verify model file
import sentencepiece as spm
import os

model_path = 'tokenizer/turkish_bpe.model'
assert os.path.exists(model_path), f"Model not found: {model_path}"

sp = spm.SentencePieceProcessor()
sp.load(model_path)

assert sp.get_piece_size() == 64000, f"Expected 64000 vocab, got {sp.get_piece_size()}"

print(f"[OK] Model loaded: {model_path}")
print(f"[OK] Vocab size: {sp.get_piece_size():,}")
print(f"[OK] pad_id={sp.pad_id()}, unk_id={sp.unk_id()}, bos_id={sp.bos_id()}, eos_id={sp.eos_id()}")
print(f"[OK] Quick encode test: {sp.encode('merhaba dunya')}")

---
## Section 4: Validate Tokenizer & Copy Artifacts to Drive

Runs 4 validation checks:
- **fertility**: avg tokens/word <= 1.4 (sampled 10,000 sentences)
- **roundtrip**: encode->decode lossless for 100 test strings
- **turkish_chars**: all 12 Turkish chars have dedicated vocabulary entries
- **special_tokens**: all 8 special token IDs are correct and distinct

**Expected exit code**: 0 (all pass)

**Expected output**:
```
[INFO] Model loaded: tokenizer/turkish_bpe.model (vocab_size=64,000)

Check              Status       Value  Threshold
------------------------------------------------
fertility          [PASS]      1.2300     1.4000
roundtrip          [PASS]    100.0000   100.0000
turkish_chars      [PASS]     12.0000    12.0000
special_tokens     [PASS]      8.0000     8.0000

All checks passed.
```

In [ ]:
!python tokenizer/validate_tokenizer.py
print(f"\nValidation exit code: {__import__('subprocess').run(['python','tokenizer/validate_tokenizer.py'], capture_output=True).returncode}")

In [ ]:
# Smoke test the Tokenizer wrapper (US3 interface)
from tokenizer import Tokenizer

tok = Tokenizer('tokenizer/turkish_bpe.model')

assert tok.vocab_size == 64000
assert tok.pad_id == 0
assert tok.unk_id == 1
assert tok.bos_id == 2
assert tok.eos_id == 3
assert tok.system_id == 4
assert tok.user_id == 5
assert tok.assistant_id == 6
assert tok.sep_id == 7

# Round-trip
test = 'merhaba dunya'
assert tok.decode(tok.encode(test)) == test, "Round-trip failed!"

# BOS/EOS
ids_with_bos_eos = tok.encode(test, add_bos=True, add_eos=True)
assert ids_with_bos_eos[0] == tok.bos_id
assert ids_with_bos_eos[-1] == tok.eos_id

# Empty string
assert tok.encode('') == []

print(f"[OK] Tokenizer: {tok}")
print(f"[OK] encode('merhaba dunya') -> {tok.encode('merhaba dunya')}")
print(f"[OK] All Tokenizer assertions passed")

In [ ]:
# Copy model artifacts to Google Drive for persistence across Colab sessions
import shutil
import os

artifacts = [
    ('tokenizer/turkish_bpe.model', f'{DRIVE_DIR}/turkish_bpe.model'),
    ('tokenizer/turkish_bpe.vocab', f'{DRIVE_DIR}/turkish_bpe.vocab'),
]

for src, dst in artifacts:
    if os.path.exists(src):
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / 1024 / 1024
        print(f"[OK] Copied {src} -> {dst} ({size_mb:.1f} MB)")
    else:
        print(f"[WARN] Artifact not found: {src} — skipping")

print(f"\nArtifacts saved to: {DRIVE_DIR}")
print("Phase 1 (Tokenizer) complete!")